# EF characterization workflow (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook switches to `ge-ef-cr` mode and walks through the core
`|e>` to `|f>` calibration steps: prepare the ge pulses, obtain EF Rabi
data, refine the EF drive frequency, and calibrate an EF half-pi pulse.

## 1. Create an `Experiment` in `ge-ef-cr` mode

`ge-ef-cr` assigns control channels in the order `ge`, `ef`, and `cr`.
Use this mode when you need direct EF control in the same session. If the
active AWG profile leaves a control port with only one channel, that port
resolves to `ge` only, so you cannot drive EF on that qubit.

In [1]:
import numpy as np

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    configuration_mode="ge-ef-cr",
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [2]:
Q0 = exp.qubit_labels[0]

print("target qubit:", Q0)

target qubit: Q00


## 2. Connect to the setup

Connect before you run any checks or EF calibrations.

In [3]:
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-ef-cr', drag_hpi_puls

## 3. Configure the `ge-ef-cr` layout

After `connect()`, call `configure()` so the control hardware applies the
`ge-ef-cr` channel layout before you run waveform checks or EF calibrations.

> [!CAUTION]
> `configure()` changes the state of the control instruments. On shared
> systems, it can affect other users who are using the same hardware. Also,
> if a control port has only one AWG channel, that port stays `ge`-only and
> EF drive is not available there.

In [4]:
exp.configure()

## 4. Check the waveform

Begin with a waveform check so readout issues do not get confused with EF-drive problems.

In [5]:
waveform_result = exp.check_waveform()

## 5. Prepare the `ge` pulse set

Measure a baseline Rabi oscillation, calibrate the half-pi pulse, and save the result before moving to EF work.

In [6]:
rabi_result = exp.obtain_rabi_params()
hpi_result = exp.calibrate_hpi_pulse()
exp.calib_note.save()

## 6. Measure the EF Rabi oscillation

Use an EF Rabi scan to locate a usable EF drive setting before the frequency calibration.

In [7]:
ef_rabi_result = exp.obtain_ef_rabi_params(
    Q0,
    time_range=np.arange(0, 201, 4),
    n_shots=1024,
)

## 7. Calibrate the EF control frequency

Run the EF frequency calibration over a relatively narrow detuning window.

In [8]:
ef_frequency_result = exp.calibrate_ef_control_frequency(
    Q0,
    detuning_range=np.linspace(-0.02, 0.02, 41),
    time_range=np.arange(0, 101, 4),
)

## 8. Update the anharmonicity and reload

Write the fitted anharmonicity to the parameter file, then reload before you re-check the EF Rabi scan.

In [9]:
# Update the fitted anharmonicity manually, then reload
exp.reload()

## 9. Re-check the EF Rabi oscillation

Run the EF Rabi measurement again to confirm that the updated frequency improved the response.

In [10]:
ef_rabi_after_frequency = exp.obtain_ef_rabi_params(
    Q0,
    time_range=np.arange(0, 201, 4),
    n_shots=1024,
)

## 10. Calibrate the EF half-pi pulse

With the EF frequency in place, calibrate the EF half-pi pulse.

In [11]:
ef_hpi_result = exp.calibrate_ef_hpi_pulse(
    Q0,
    n_rotations=1,
)

## 11. Save the calibration note

Save the updated note so later EF workflows start from the calibrated parameters.

In [12]:
exp.calib_note.save()